# Laboratori 2 · Ushtrime Shtesë — Analiza Asimptotike: Numërimi i Veprimeve dhe Përkufizimi i O(·)

> **Lidhja me leksionin:** *Leksioni 2 — Analiza asimptotike* (slides 3–25) dhe hyrja e
> *Leksionit 3* (slide 2: "Sa i shpejtë është Insertion Sort?", slide 21: pseudokodi i `Merge`).
>
> Laboratori kryesor e paraqiti Big O me grafikë. Këtu e përdorim **përkufizimin matematik**:
> numërojmë veprime, gjejmë konstantet $c$ dhe $n_0$, dhe implementojmë `Merge` saktë.
> Nuk kërkon Pygame.

## Kujtues teorik — Nga sekondat te O(·)

**Pse jo sekonda?** "I njëjti" algoritëm është më i shpejtë ose më i ngadaltë në varësi të gjuhës,
të programuesit dhe të kompjuterit (slide 4). Prandaj **numërojmë veprime elementare** dhe pastaj
shohim vetëm **rendin e rritjes**.

### Përkufizimi matematik (slide 15)

$$
T(n) = O(g(n)) \iff \exists\, c > 0,\ n_0 > 0 \ \text{të tilla që}\ \forall n \ge n_0:\ T(n) \le c\cdot g(n)
$$

- **"ekzistojnë"** $c$ dhe $n_0$ — mjafton t'i gjejmë **një herë**; nuk ka zgjedhje "ekzakte" (slide 19).
- **"për çdo"** $n \ge n_0$ — pabarazia duhet të vlejë **përgjithmonë** pas $n_0$.

**Shembulli i leksionit:** $T(n) = 2n^2 + 10$, $g(n) = n^2$.
Me $c = 3$: $\;2n^2 + 10 \le 3n^2 \iff n^2 \ge 10 \iff n \ge 4$, pra $n_0 = 4$.
Me $c = 7$ mjafton $n_0 = 2$.

### Pro dhe kundër (slide 10)
| Pro | Kundër |
|---|---|
| E pavarur nga platforma dhe gjuha | Ka kuptim vetëm për $n$ shumë të mëdha |
| Thjeshton analizën dhe krahasimin | A është $10^9 \cdot n$ "më e mirë" se $n^2$?! |

In [1]:
# ── Mjeti i kontrollit ──────────────────────────────────────────────
# Ekzekutojeni këtë qelizë një herë, në fillim. Pas çdo ushtrimi, qeliza
# "kontrollo(...)" teston funksionin tuaj dhe ju tregon nëse është gati.

def kontrollo(titulli, testet):
    try:
        testet()
    except NotImplementedError:
        print(f"⏳ {titulli}: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.")
    except AssertionError as e:
        print(f"✗ {titulli}: {e}")
    except Exception as e:
        print(f"✗ {titulli}: kodi u ndal me gabim  {type(e).__name__}: {e}")
    else:
        print(f"✓ {titulli}: të gjitha testet kaluan!")

## Ushtrimi 1 — Tabela e numërimit të rreshtave (slide 5)

Leksioni numëron sa herë ekzekutohet secili rresht i `InsertionSort(A, n)`.
Funksioni `numero_rreshtat` më poshtë e bën këtë automatikisht. Me $k_i$ shënojmë numrin e
spostimeve në iteracionin $i$:

| Rreshti | Numri i ekzekutimeve $T_i$ |
|:-:|---|
| 1 `for` | $n$ (përfshirë kontrollin e fundit që del nga cikli) |
| 2, 3, 7 | $n - 1$ secili |
| 4 `while` | $\sum_i (k_i + 1)$ — çdo spostim + kontrolli që e ndal ciklin |
| 5, 6 | $\sum_i k_i$ secili |

In [2]:
def numero_rreshtat(A):
    A = list(A)
    n = len(A)
    T = {r: 0 for r in range(1, 8)}
    for i in range(1, n):
        T[1] += 1
        aktual = A[i];  T[2] += 1
        j = i - 1;      T[3] += 1
        while True:
            T[4] += 1
            if not (j >= 0 and A[j] > aktual):
                break
            A[j + 1] = A[j]; T[5] += 1
            j = j - 1;       T[6] += 1
        A[j + 1] = aktual;   T[7] += 1
    T[1] += 1 if n > 0 else 0       # kontrolli i fundit i `for`
    return T


import random
n = 10
raste = {"e renditur": list(range(n)), "e rastit": random.Random(7).sample(range(n), n),
         "zbritëse": list(range(n, 0, -1))}
print(f"{'Rreshti':>8} | " + " | ".join(f"{k:>11}" for k in raste))
tabelat = {k: numero_rreshtat(v) for k, v in raste.items()}
for r in range(1, 8):
    print(f"{r:>8} | " + " | ".join(f"{tabelat[k][r]:>11}" for k in raste))
print(f"{'Total':>8} | " + " | ".join(f"{sum(tabelat[k].values()):>11}" for k in raste))

 Rreshti |  e renditur |    e rastit |    zbritëse
       1 |          10 |          10 |          10
       2 |           9 |           9 |           9
       3 |           9 |           9 |           9
       4 |           9 |          31 |          54
       5 |           0 |          22 |          45
       6 |           0 |          22 |          45
       7 |           9 |           9 |           9
   Total |          46 |         112 |         181


**Detyra:** nga tabela nxirrni **formulat e mbyllura** për totalin $T(n) = \sum_i T_i$:

- `T_me_i_mire(n)` — lista e renditur ($k_i = 0$ për çdo $i$);
- `T_me_i_keq(n)` — lista zbritëse ($k_i = i$, pra $\sum k_i = \frac{n(n-1)}{2}$).

Kontrolli i krahason me `numero_rreshtat` për $n = 1, \dots, 60$.

In [3]:
def T_me_i_mire(n):
    # PLOTESO: n + 3(n−1) + (n−1) + 0 + 0 = ?
    raise NotImplementedError


def T_me_i_keq(n):
    # PLOTESO: shtoni kontributin e rreshtave 4, 5, 6 kur Σk_i = n(n−1)/2
    raise NotImplementedError


def _testet():
    for n in range(1, 61):
        m = sum(numero_rreshtat(range(n)).values())
        assert T_me_i_mire(n) == m, f"n={n}: rasti më i mirë ka {m} ekzekutime, formula jep {T_me_i_mire(n)}"
        k = sum(numero_rreshtat(range(n, 0, -1)).values())
        assert T_me_i_keq(n) == k, f"n={n}: rasti më i keq ka {k} ekzekutime, formula jep {T_me_i_keq(n)}"

kontrollo("Ushtrimi 1 — Formulat e numërimit", _testet)

⏳ Ushtrimi 1 — Formulat e numërimit: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


> **Krahasoni me leksionin.** Slide 5 numëron *veprime elementare* (p.sh. rreshti 3 `j = i−1` = një zbritje
> + një vlerëdhënie = 2 veprime) dhe merr $\;7n - 6 \le T(n) \le 3{,}5n^2 + 1{,}5n - 4$.
> Ne numëruam *rreshta* dhe morëm konstante të tjera. **Rendi i rritjes është i njëjtë:** $n$ në rastin më të mirë,
> $n^2$ në rastin më të keq. Pikërisht kjo është arsyeja pse O(·) i injoron konstantet.

## Ushtrimi 2 — Gjeni c dhe n₀

**Detyra:**
1. `verifiko_O(T, g, c, n0, n_max)` — kthen `True` nëse $T(n) \le c\cdot g(n)$ për **çdo** $n$ me $n_0 \le n \le n_{max}$.
2. `gjej_n0(T, g, c, n_max)` — kthen $n_0$ **më të vogël** në $[1, n_{max}]$ që e bën `verifiko_O` të vërtetë,
   ose `None` nëse pabarazia dështon edhe për $n = n_{max}$.

> **Kujdes:** kompjuteri kontrollon vetëm deri në $n_{max}$. Kjo **nuk** është vërtetim — vërtetimi
> është algjebrik (si në kujtuesin teorik). Por është mënyrë e shkëlqyer për të gjetur kandidatët.

In [4]:
def verifiko_O(T, g, c, n0, n_max=10_000):
    # PLOTESO: all(...) mbi n nga n0 deri n_max
    raise NotImplementedError


def gjej_n0(T, g, c, n_max=10_000):
    # PLOTESO: gjeni n-në e fundit ku pabarazia dështon (nëse ka)
    raise NotImplementedError

In [5]:
T  = lambda n: 2 * n**2 + 10
g  = lambda n: n**2

def _testet():
    assert verifiko_O(T, g, 3, 4), "slide 18: c=3, n0=4 duhet të funksionojë"
    assert not verifiko_O(T, g, 3, 3), "c=3, n0=3 NUK funksionon: 2·9+10 = 28 > 27"
    assert verifiko_O(T, g, 7, 2), "slide 19: c=7, n0=2 duhet të funksionojë"
    assert verifiko_O(lambda n: n, g, 1, 1), "slide 20: n = O(n²) me c=1, n0=1"
    assert gjej_n0(T, g, 3) == 4, f"për c=3 pritej n0=4, u mor {gjej_n0(T, g, 3)}"
    assert gjej_n0(T, g, 7) == 2, f"për c=7 pritej n0=2, u mor {gjej_n0(T, g, 7)}"
    assert gjej_n0(T, g, 12) == 1, "për c=12 vlen që nga n=1"
    assert gjej_n0(lambda n: n**2, lambda n: n, 1000, 5000) is None, \
        "n² ≤ 1000·n dështon për çdo n > 1000 — duhet None"

kontrollo("Ushtrimi 2 — c dhe n0", _testet)

⏳ Ushtrimi 2 — c dhe n0: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


### Grafiku i slides 18–19

In [6]:
import plotly.graph_objects as go

ns = [x / 10 for x in range(5, 61)]
fig = go.Figure()
fig.add_scatter(x=ns, y=[2 * n**2 + 10 for n in ns], name="T(n) = 2n² + 10", line=dict(width=3))
fig.add_scatter(x=ns, y=[3 * n**2 for n in ns], name="3·g(n) = 3n²")
fig.add_scatter(x=ns, y=[7 * n**2 for n in ns], name="7·g(n) = 7n²")
fig.add_scatter(x=ns, y=[n**2 for n in ns], name="g(n) = n²", line=dict(dash="dot"))
fig.add_vline(x=4, line_dash="dash", annotation_text="n₀ = 4 (c = 3)")
fig.add_vline(x=2, line_dash="dash", annotation_text="n₀ = 2 (c = 7)", annotation_position="top left")
fig.update_layout(title="T(n) = O(n²): pas n₀, T(n) qëndron nën c·g(n)",
                  xaxis_title="n", yaxis_title="vlera", height=440)
fig.show()

## Ushtrimi 3 — Rendi i rritjes së funksioneve

Rregullat praktike që dalin nga përkufizimi (slides 8–9):
**(1)** injorojmë faktorët konstantë, **(2)** mbajmë vetëm termin me rend më të lartë rritjeje.

**Detyra:** plotësoni fjalorin `pergjigjet` me klasën O(·) të çdo funksioni.
Përdorni vetëm këto përgjigje: `"1"`, `"log n"`, `"n"`, `"n log n"`, `"n^2"`, `"n^3"`, `"2^n"`.

In [7]:
pergjigjet = {
    # PLOTESO: zëvendësoni çdo "?" me klasën përkatëse
    "3n² + 100n + 7":        "?",
    "5n·log n + 20n":        "?",
    "1000":                  "?",
    "n + log n":             "?",
    "2ⁿ + n³":               "?",
    "n(n−1)/2":              "?",
    "7n − 6":                "?",
    "3,5n² + 1,5n − 4":      "?",
    "log(n²)":               "?",
    "(n + 1)³":              "?",
}

In [8]:
import hashlib

def _h(s):
    return hashlib.sha1(s.replace(" ", "").encode()).hexdigest()[:6]

# Përgjigjet e sakta ruhen si hash, që të mos i shihni para se të provoni :)
_sakte = {"3n² + 100n + 7": "a0a0ec", "5n·log n + 20n": "2a6ebe", "1000": "356a19",
          "n + log n": "d1854c", "2ⁿ + n³": "b847b0", "n(n−1)/2": "a0a0ec", "7n − 6": "d1854c",
          "3,5n² + 1,5n − 4": "a0a0ec", "log(n²)": "3defee", "(n + 1)³": "31330e"}

def _testet():
    if all(v == "?" for v in pergjigjet.values()):
        raise NotImplementedError
    gabime = [f for f, h in _sakte.items() if _h(str(pergjigjet.get(f, "?"))) != h]
    assert not gabime, f"{len(gabime)} përgjigje të gabuara: {', '.join(gabime)}"

kontrollo("Ushtrimi 3 — Klasat O(·)", _testet)

⏳ Ushtrimi 3 — Klasat O(·): ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


> **Pyetje për `log(n²)`:** pse $\log(n^2) = 2\log n$ ka të njëjtin rend si $\log n$?
> Po $\log_{10} n$ kundrejt $\log_2 n$?

## Ushtrimi 4 — Sa kohë do të duhej? (Leksioni 3, slide 2)

Supozojmë $10^9$ veprime në sekondë. Tabela më poshtë rindërton slide-in e leksionit dhe shton kolonën
për $n \log n$ — çmimi i Merge Sort-it.

In [9]:
import math

def formato(sekonda):
    if sekonda < 1:            return f"{sekonda:.0e} sekonda"
    if sekonda < 3600:         return f"{sekonda:,.0f} sekonda"
    if sekonda < 86400 * 365:  return f"{sekonda / 86400:,.1f} ditë"
    return f"{sekonda / (86400 * 365):,.0f} vite"

print(f"{'n':>16} | {'n² veprime':>18} | {'n·log₂n veprime':>18}")
print("-" * 58)
for n, emri in [(100, ""), (1_000, ""), (1_000_000, ""), (3_000_000_000, "(Facebook MAU)")]:
    print(f"{n:>16,} | {formato(n**2 / 1e9):>18} | {formato(n * math.log2(n) / 1e9):>18}  {emri}")

               n |         n² veprime |    n·log₂n veprime
----------------------------------------------------------
             100 |      1e-05 sekonda |      7e-07 sekonda  
           1,000 |      1e-03 sekonda |      1e-05 sekonda  
       1,000,000 |      1,000 sekonda |      2e-02 sekonda  
   3,000,000,000 |           285 vite |         94 sekonda  (Facebook MAU)


> **Diskutim — "A është $10^9 \cdot n$ më e mirë se $n^2$?"** (slide 10). Asimptotikisht **po**: $10^9 n = O(n^2)$
> por jo anasjelltas. Në praktikë, $10^9 n < n^2$ vetëm kur $n > 10^9$. Për çdo input më të vogël,
> algoritmi "më i keq" asimptotikisht është më i shpejtë! Jepni një shembull nga jeta reale ku kjo ka rëndësi.

## Ushtrimi 5 — `Merge(L, R)` saktë

Pseudokodi i leksionit (Leksioni 3, slide 21):

```
Merge(L, R)          # Supozojmë se L dhe R janë të renditura
1   S = listë me gjatësi n
2   i = 0, j = 0
3   for k ⟵ 0, …, n−1:
4       if L[i] < R[j]:
5           S[k] = L[i];  i = i + 1
7       else:
8           S[k] = R[j];  j = j + 1
10  return S
```

Ky pseudokod është **i thjeshtuar** për të treguar idenë. Përkthyer fjalë për fjalë në Python, ai dështon:

In [10]:
def merge_fjale_per_fjale(L, R):
    n = len(L) + len(R)
    S = [None] * n
    i = j = 0
    for k in range(n):
        if L[i] < R[j]:
            S[k] = L[i]; i += 1
        else:
            S[k] = R[j]; j += 1
    return S

try:
    merge_fjale_per_fjale([1, 2], [3, 4])
except IndexError as e:
    print("IndexError:", e, "— çfarë ndodh kur i == len(L)?")

IndexError: list index out of range — çfarë ndodh kur i == len(L)?


**Detyra:** implementoni `merge(L, R)` që trajton rastin kur njëra listë **mbaron** para tjetrës.
Dy mënyra të njohura:
- kur `i == len(L)` merrni nga `R` (dhe anasjelltas), ose
- shtoni në fund të `L` dhe `R` një **rojë** (*sentinel*) `float('inf')`, si te libri CLRS.

Kontrolli verifikon edhe **numrin e krahasimeve**: `Merge` duhet të bëjë **të shumtën** $|L| + |R| - 1$
krahasime mes elementeve — pra kohë $O(n)$ (slide 36).

In [11]:
def merge(L, R):
    # PLOTESO: ktheni një listë të re S me të gjitha elementet e L dhe R, të renditura
    raise NotImplementedError

In [12]:
class Numer:
    # Numër që numëron sa herë krahasohet me një tjetër.
    krahasime = 0
    def __init__(self, v): self.v = v
    def _k(self, o):
        Numer.krahasime += 1
        return o.v if isinstance(o, Numer) else o
    def __lt__(self, o): return self.v <  self._k(o)
    def __le__(self, o): return self.v <= self._k(o)
    def __gt__(self, o): return self.v >  self._k(o)
    def __ge__(self, o): return self.v >= self._k(o)
    def __eq__(self, o): return self.v == (o.v if isinstance(o, Numer) else o)


def _testet():
    assert merge([1, 4, 9], [2, 3, 10, 11]) == [1, 2, 3, 4, 9, 10, 11]
    assert merge([1, 2], [3, 4]) == [1, 2, 3, 4], "rasti ku L mbaron e para"
    assert merge([3, 4], [1, 2]) == [1, 2, 3, 4], "rasti ku R mbaron e para"
    assert merge([], [5, 6]) == [5, 6] and merge([5, 6], []) == [5, 6], "lista bosh"
    assert merge([1, 3, 3], [2, 3]) == [1, 2, 3, 3, 3], "vlera të përsëritura"
    rng = random.Random(0)
    for _ in range(200):
        L = sorted(rng.sample(range(1000), rng.randint(0, 30)))
        R = sorted(rng.sample(range(1000), rng.randint(0, 30)))
        Numer.krahasime = 0
        S = merge([Numer(x) for x in L], [Numer(x) for x in R])
        assert [s.v for s in S] == sorted(L + R), f"rezultat i gabuar për L={L}, R={R}"
        if L and R:
            assert Numer.krahasime <= len(L) + len(R) - 1, \
                f"{Numer.krahasime} krahasime për n={len(L) + len(R)} — Merge duhet të jetë O(n)"

kontrollo("Ushtrimi 5 — Merge", _testet)

⏳ Ushtrimi 5 — Merge: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Punë në Klasë

**Problemi 1.** Plotësoni tabelën e numërimit të rreshtave të `InsertionSort` për $A = [5, 4, 3, 2, 1]$
(rasti më i keq, $n = 5$). Verifikoni që totali është i barabartë me `T_me_i_keq(5)`.

| Rreshti | 1 | 2 | 3 | 4 | 5 | 6 | 7 | Total |
|---|---|---|---|---|---|---|---|---|
| $T_i$ | | | | | | | | |

**Problemi 2.** Vërtetoni me përkufizim që $5n^3 + 2n = O(n^3)$. Jepni $c$ dhe $n_0$ dhe argumentoni algjebrikisht.

**Problemi 3.** Vërtetoni që $n^2 \neq O(n)$. *Ndihmë: supozoni të kundërtën — ekzistojnë $c, n_0$ me $n^2 \le c\,n$
për çdo $n \ge n_0$. Çfarë ndodh për $n = \max(n_0, \lceil c \rceil + 1)$?*

**Problemi 4.** Renditni sipas rendit të rritjes (nga më i ngadalti te më i shpejti):
$n\log n,\ n^2,\ \log n,\ 2^n,\ \sqrt n,\ n,\ 1000,\ n^3$.

**Problemi 5.** Ekzekutoni me dorë `merge([2, 5, 8], [1, 6, 7, 9])`. Shkruani pas çdo hapi vlerat e `i`, `j` dhe `S`.
Sa krahasime u bënë?

## Pyetje Reflektuese

1. Pse dy mënyra të ndryshme numërimi (rreshta kundrejt veprimeve elementare) japin konstante të ndryshme,
   por të njëjtin O(·)? Çfarë do të thotë kjo për mënyrën si i krahasojmë algoritmet?
2. Nëse $T(n) = O(n^2)$, a është gjithashtu $T(n) = O(n^3)$? Po $O(n)$? Pse $O$ quhet **kufi i sipërm**?
3. Pseudokodi i `Merge` në slide nuk trajton fundin e listave. Pse mendoni se leksioni e thjeshtoi kështu?
4. Merge Sort përdor $O(n)$ kujtesë shtesë; Insertion Sort jo. Kur do ta zgjidhnit Insertion Sort?